# 02 - Bronze Data Validation

## Purpose

This notebook validates the quality and structure of all Bronze layer census tables before they are transformed into the Silver layer.

### Validation Checks

- Row Count
- Duplicate Records
- Null Values
- Gender Consistency
- Rural/Urban Consistency
- Hierarchical Structure
- Geographic Structure
- Industrial Table Integrity

No data is modified in this notebook.

The notebook is intended to provide confidence that the Bronze layer has been ingested correctly before further processing.

### 1 — Imports

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    when,
    sum,
    lit
)
from functools import reduce

## Helper Functions

The following reusable functions are used throughout this notebook to avoid duplicated validation logic.

### 2 — Generic Information

In [0]:
def basic_information(df, table_name):

    print("=" * 70)
    print(table_name.upper())
    print("=" * 70)

    print(f"Rows    : {df.count()}")
    print(f"Columns : {len(df.columns)}")

    print()

### 3 — Duplicate Validation

In [0]:
def duplicate_validation(df):

    total_rows = df.count()
    distinct_rows = df.distinct().count()

    print("Duplicate Validation")
    print("--------------------")

    print(f"Total Rows     : {total_rows}")
    print(f"Distinct Rows  : {distinct_rows}")
    print(f"Duplicate Rows : {total_rows-distinct_rows}")

    print()

### 4 — Null Validation

In [0]:
def null_validation(df):

    print("Null Validation")
    print("----------------")

    has_null=False

    for c in df.columns:

        n=df.filter(col(c).isNull()).count()

        if n>0:

            has_null=True
            print(f"{c:<60}{n}")

    if not has_null:
        print("No null values found.")

    print()

### 5 — Blank Row Validation

In [0]:
def blank_row_validation(df):

    blank=df.filter(

        reduce(

            lambda x,y:x & y,

            [col(c).isNull() for c in df.columns]

        )

    )

    print("Completely Blank Rows")
    print("---------------------")

    print(blank.count())

    if blank.count()>0:

        blank.show(truncate=False)

    print()

### 6 — Gender Validation

In [0]:
def gender_validation(df):

    errors=df.filter(

        col("Total___Persons")!=

        (

            col("Total___Males")+

            col("Total___Females")

        )

    )

    print("Gender Validation")
    print("-----------------")

    print(f"Invalid Rows : {errors.count()}")

    print()

### 7 — Rural Urban Validation

In [0]:
def rural_urban_validation(df):

    errors=df.filter(

        col("Total___Persons")!=

        (

            col("Rural___Persons")+

            col("Urban___Persons")

        )

    )

    print("Rural / Urban Validation")

    print("------------------------")

    print(f"Invalid Rows : {errors.count()}")

    print()

### 8 — Population Consistency Validation

In [0]:
def industrial_gender_validation(df):

    errors=df.filter(

        col("Industrial_Category___Total___Persons")

        !=

        (

            col("Industrial_Category___Total___Males")

            +

            col("Industrial_Category___Total___Females")

        )

    )

    print("Industrial Gender Validation")

    print("----------------------------")

    print(f"Invalid Rows : {errors.count()}")

    print()

### 9 — Hierarchy Summary


In [0]:
def hierarchy_summary(df):

    print("Division Summary")

    df.groupBy("Division").count().orderBy("Division").show()

    print()

    print("Subdivision Summary")

    df.groupBy(

        "Division",

        "Sub_Division"

    ).count().orderBy(

        "Division",

        "Sub_Division"

    ).show()

    print()

### 10 — Geographic Summary

In [0]:
def geography_summary(df):

    print("Geographic Summary")

    df.groupBy(

        "District_Code"

    ).count().orderBy(

        "District_Code"

    ).show()

    print()

### Validate ST Main Workers

In [0]:
df_st=spark.read.table("pyspark_dbt.bronze.st_main_workers")

In [0]:
basic_information(df_st,"ST Main Workers")

duplicate_validation(df_st)

null_validation(df_st)

blank_row_validation(df_st)

gender_validation(df_st)

rural_urban_validation(df_st)

hierarchy_summary(df_st)

geography_summary(df_st)

ST MAIN WORKERS
Rows    : 10824
Columns : 18

Duplicate Validation
--------------------
Total Rows     : 10824
Distinct Rows  : 10824
Duplicate Rows : 0

Null Validation
----------------
No null values found.

Completely Blank Rows
---------------------
0

Gender Validation
-----------------
Invalid Rows : 0

Rural / Urban Validation
------------------------
Invalid Rows : 0

Division Summary
+--------+-----+
|Division|count|
+--------+-----+
|       0|   31|
|       1|  987|
|       2| 1402|
|       3| 1604|
|       4|  698|
|       5|  745|
|       6|  458|
|       7| 1978|
|       8| 1788|
|       9|  943|
|       X|  190|
+--------+-----+


Subdivision Summary
+--------+------------+-----+
|Division|Sub_Division|count|
+--------+------------+-----+
|       0|          00|   31|
|       1|          00|   31|
|       1|          11|  275|
|       1|          12|  604|
|       1|          13|   77|
|       2|          00|   31|
|       2|          21|  372|
|       2|          22|  22

### Validate SC Main Workers

In [0]:
df_sc=spark.read.table("pyspark_dbt.bronze.sc_main_workers")

In [0]:
basic_information(df_sc,"SC Main Workers")

duplicate_validation(df_sc)

null_validation(df_sc)

blank_row_validation(df_sc)

gender_validation(df_sc)

rural_urban_validation(df_sc)

hierarchy_summary(df_sc)

geography_summary(df_sc)

SC MAIN WORKERS
Rows    : 11677
Columns : 18

Duplicate Validation
--------------------
Total Rows     : 11677
Distinct Rows  : 11677
Duplicate Rows : 0

Null Validation
----------------
No null values found.

Completely Blank Rows
---------------------
0

Gender Validation
-----------------
Invalid Rows : 0

Rural / Urban Validation
------------------------
Invalid Rows : 0

Division Summary
+--------+-----+
|Division|count|
+--------+-----+
|       0|   31|
|       1| 1040|
|       2| 1574|
|       3| 1796|
|       4|  725|
|       5|  783|
|       6|  459|
|       7| 2189|
|       8| 1897|
|       9|  993|
|       X|  190|
+--------+-----+


Subdivision Summary
+--------+------------+-----+
|Division|Sub_Division|count|
+--------+------------+-----+
|       0|          00|   31|
|       1|          00|   31|
|       1|          11|  303|
|       1|          12|  622|
|       1|          13|   84|
|       2|          00|   31|
|       2|          21|  404|
|       2|          22|  26

### Validate Industrial Main Workers

In [0]:
df_ind_main=spark.read.table("pyspark_dbt.bronze.industrial_main_workers")

In [0]:
basic_information(df_ind_main,"Industrial Main Workers")

duplicate_validation(df_ind_main)

null_validation(df_ind_main)

blank_row_validation(df_ind_main)

industrial_gender_validation(df_ind_main)

hierarchy_summary(df_ind_main)

geography_summary(df_ind_main)

print("Area Classification")

df_ind_main.groupBy(

    "Total_Rural_Urban"

).count().show()

INDUSTRIAL MAIN WORKERS
Rows    : 3560
Columns : 62

Duplicate Validation
--------------------
Total Rows     : 3560
Distinct Rows  : 3559
Duplicate Rows : 1

Null Validation
----------------
Table_Name                                                  2
State_Code                                                  5
District_Code                                               5
Area_Name                                                   5
Division                                                    5
Sub_Division                                                5
NCO_Name                                                    5
Total_Rural_Urban                                           5
Industrial_Category___Total___Persons                       5
Industrial_Category___Total___Males                         5
Industrial_Category___Total___Females                       5
Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Persons5
Industrial_Category___A__Excluding_Cultivat

### Validate Industrial Marginal Workers

In [0]:
df_ind_marg=spark.read.table("pyspark_dbt.bronze.industrial_marginal_workers")

In [0]:
basic_information(df_ind_marg,"Industrial Marginal Workers")

duplicate_validation(df_ind_marg)

null_validation(df_ind_marg)

blank_row_validation(df_ind_marg)

industrial_gender_validation(df_ind_marg)

hierarchy_summary(df_ind_marg)

geography_summary(df_ind_marg)

print("Area Classification")

df_ind_marg.groupBy(

    "Total_Rural_Urban"

).count().show()

INDUSTRIAL MARGINAL WORKERS
Rows    : 3507
Columns : 62

Duplicate Validation
--------------------
Total Rows     : 3507
Distinct Rows  : 3507
Duplicate Rows : 0

Null Validation
----------------
No null values found.

Completely Blank Rows
---------------------
0

Industrial Gender Validation
----------------------------
Invalid Rows : 0

Division Summary
+--------+-----+
|Division|count|
+--------+-----+
|       0|   93|
|       1|  336|
|       2|  465|
|       3|  465|
|       4|  279|
|       5|  279|
|       6|  195|
|       7|  465|
|       8|  372|
|       9|  372|
|       X|  186|
+--------+-----+


Subdivision Summary
+--------+------------+-----+
|Division|Sub_Division|count|
+--------+------------+-----+
|       0|          00|   93|
|       1|          00|   93|
|       1|          11|   93|
|       1|          12|   93|
|       1|          13|   57|
|       2|          00|   93|
|       2|          21|   93|
|       2|          22|   93|
|       2|          23|   93|
|   

### Industrial Validation

In [0]:
from pyspark.sql.functions import col

def validate_industrial_gender_totals(df):
    """
    Validates every Industrial_Category_*_Persons column against
    its corresponding Males + Females columns.

    Example:
    A Persons = A Males + A Females
    B Persons = B Males + B Females
    ...
    """

    print("=" * 70)
    print("INDUSTRIAL CATEGORY VALIDATION")
    print("=" * 70)

    person_columns = [
        c for c in df.columns
        if c.endswith("Persons")
    ]

    total_errors = 0

    for persons_col in person_columns:

        males_col = persons_col.replace("Persons", "Males")
        females_col = persons_col.replace("Persons", "Females")

        if males_col in df.columns and females_col in df.columns:

            errors = df.filter(
                col(persons_col) != (
                    col(males_col) + col(females_col)
                )
            ).count()

            print(f"{persons_col}")
            print(f"Errors : {errors}")
            print()

            total_errors += errors

    print("=" * 70)
    print(f"TOTAL VALIDATION ERRORS : {total_errors}")

In [0]:
validate_industrial_gender_totals(df_ind_main)

INDUSTRIAL CATEGORY VALIDATION
Industrial_Category___Total___Persons
Errors : 0

Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Persons
Errors : 0

Industrial_Category___B___Persons
Errors : 0

Industrial_Category___C___HHI___Persons
Errors : 0

Industrial_Category___C___Non_HHI___Persons
Errors : 0

Industrial_Category___D___E___Persons
Errors : 0

Industrial_Category___F___Persons
Errors : 0

Industrial_Category___G___HHI___Persons
Errors : 0

Industrial_Category___G___Non_HHI___Persons
Errors : 0

Industrial_Category___H___Persons
Errors : 0

Industrial_Category___I___Persons
Errors : 0

Industrial_Category___J___HHI___Persons
Errors : 0

Industrial_Category___J___Non_HHI___Persons
Errors : 0

Industrial_Category___K_to_M___Persons
Errors : 0

Industrial_Category___N_to_O___Persons
Errors : 0

Industrial_Category___P_to_Q___Persons
Errors : 0

Industrial_Category___R_to_U___HHI___Persons
Errors : 0

Industrial_Category___R_to_U___Non_HHI___Persons
Errors :

In [0]:
validate_industrial_gender_totals(df_ind_marg)

INDUSTRIAL CATEGORY VALIDATION
Industrial_Category___Total___Persons
Errors : 0

Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Persons
Errors : 0

Industrial_Category___B___Persons
Errors : 0

Industrial_Category___C___HHI___Persons
Errors : 0

Industrial_Category___C___Non_HHI___Persons
Errors : 0

Industrial_Category___D___E___Persons
Errors : 0

Industrial_Category___F___Persons
Errors : 0

Industrial_Category___G___HHI___Persons
Errors : 0

Industrial_Category___G___Non_HHI___Persons
Errors : 0

Industrial_Category___H___Persons
Errors : 0

Industrial_Category___I___Persons
Errors : 0

Industrial_Category___J___HHI___Persons
Errors : 0

Industrial_Category___J___Non_HHI___Persons
Errors : 0

Industrial_Category___K_to_M___Persons
Errors : 0

Industrial_Category___N_to_O___Persons
Errors : 0

Industrial_Category___P_to_Q___Persons
Errors : 0

Industrial_Category___R_to_U___HHI___Persons
Errors : 0

Industrial_Category___R_to_U___Non_HHI___Persons
Errors :

# Validation Summary

## Key Findings

- Bronze tables successfully loaded.
- Duplicate records identified (if any).
- Blank rows identified (if any).
- Null values summarized.
- Gender totals validated.
- Rural and Urban totals validated.
- Occupational hierarchy verified.
- Geographic hierarchy verified.
- Industrial area categories verified.

The validated Bronze tables are now ready for Silver layer transformations.

| Table                       | Duplicate Rows |                               Blank Rows | Null Values | Status              |
| --------------------------- | -------------: | ---------------------------------------: | ----------: | ------------------- |
| ST Main Workers             |              0 |                                        0 |           0 | Pass                |
| SC Main Workers             |              0 |                                        0 |           0 | Pass                |
| Industrial Marginal Workers |              0 |                                        0 |           0 | Pass                |
| Industrial Main Workers     |              1 | 2 (or 5 null rows pending investigation) |     Present | Clean before Silver |


In [0]:
display(
    df_ind_main.filter(col("Division").isNull())
)

Table_Name,State_Code,District_Code,Area_Name,Division,Sub_Division,NCO_Name,Total_Rural_Urban,Industrial_Category___Total___Persons,Industrial_Category___Total___Males,Industrial_Category___Total___Females,Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Persons,Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Males,Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Females,Industrial_Category___B___Persons,Industrial_Category___B___Males,Industrial_Category___B___Females,Industrial_Category___C___HHI___Persons,Industrial_Category___C___HHI___Males,Industrial_Category___C___HHI___Females,Industrial_Category___C___Non_HHI___Persons,Industrial_Category___C___Non_HHI___Males,Industrial_Category___C___Non_HHI___Females,Industrial_Category___D___E___Persons,Industrial_Category___D___E___Males,Industrial_Category___D___E___Females,Industrial_Category___F___Persons,Industrial_Category___F___Males,Industrial_Category___F___Females,Industrial_Category___G___HHI___Persons,Industrial_Category___G___HHI___Males,Industrial_Category___G___HHI___Females,Industrial_Category___G___Non_HHI___Persons,Industrial_Category___G___Non_HHI___Males,Industrial_Category___G___Non_HHI___Females,Industrial_Category___H___Persons,Industrial_Category___H___Males,Industrial_Category___H___Females,Industrial_Category___I___Persons,Industrial_Category___I___Males,Industrial_Category___I___Females,Industrial_Category___J___HHI___Persons,Industrial_Category___J___HHI___Males,Industrial_Category___J___HHI___Females,Industrial_Category___J___Non_HHI___Persons,Industrial_Category___J___Non_HHI___Males,Industrial_Category___J___Non_HHI___Females,Industrial_Category___K_to_M___Persons,Industrial_Category___K_to_M___Males,Industrial_Category___K_to_M___Females,Industrial_Category___N_to_O___Persons,Industrial_Category___N_to_O___Males,Industrial_Category___N_to_O___Females,Industrial_Category___P_to_Q___Persons,Industrial_Category___P_to_Q___Males,Industrial_Category___P_to_Q___Females,Industrial_Category___R_to_U___HHI___Persons,Industrial_Category___R_to_U___HHI___Males,Industrial_Category___R_to_U___HHI___Females,Industrial_Category___R_to_U___Non_HHI___Persons,Industrial_Category___R_to_U___Non_HHI___Males,Industrial_Category___R_to_U___Non_HHI___Females
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Note: Table based on estimates.,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
INDUSTRIAL CATEGORIES :,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"A- Agriculture, Forestry and Fishing; B - Mining and Quarrying; C - Manufacturing; D - Electricity, Gas, Steam, and Air Conditioning Supply; E- Water Supply; (Sewarage, Waste Management and remediation activities); F - Construction; G- Wholesale and Retail Trade (Repair of motor vehicles and motor cycles); H- Transportation and Storage; I- Accomodation and food service activities;

In [0]:
df_ind_main.filter(col("Division").isNull()).count()

5

### Metadata Row Investigation

In [0]:
from functools import reduce
from pyspark.sql.functions import col

def investigate_invalid_rows(df):

    print("=" * 70)
    print("INVALID ROW INVESTIGATION")
    print("=" * 70)

    invalid = df.filter(
        reduce(
            lambda x, y: x | y,
            [col(c).isNull() for c in df.columns]
        )
    )

    print(f"Rows containing at least one NULL : {invalid.count()}")

    invalid.show(truncate=False)

In [0]:
display(investigate_invalid_rows(df_ind_marg))

INVALID ROW INVESTIGATION
Rows containing at least one NULL : 0
+----------+----------+-------------+---------+--------+------------+--------+-----------------+-------------------------------------+-----------------------------------+-------------------------------------+---------------------------------------------------------------------------------+-------------------------------------------------------------------------------+---------------------------------------------------------------------------------+---------------------------------+-------------------------------+---------------------------------+---------------------------------------+-------------------------------------+---------------------------------------+-------------------------------------------+-----------------------------------------+-------------------------------------------+-------------------------------------+-----------------------------------+-------------------------------------+------------------------

In [0]:
df_ind_main_clean = df_ind_main.filter(
    col("Table_Name").isNotNull()
)

In [0]:
display(
    df_ind_main.filter(col("Division").isNull())
)

Table_Name,State_Code,District_Code,Area_Name,Division,Sub_Division,NCO_Name,Total_Rural_Urban,Industrial_Category___Total___Persons,Industrial_Category___Total___Males,Industrial_Category___Total___Females,Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Persons,Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Males,Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Females,Industrial_Category___B___Persons,Industrial_Category___B___Males,Industrial_Category___B___Females,Industrial_Category___C___HHI___Persons,Industrial_Category___C___HHI___Males,Industrial_Category___C___HHI___Females,Industrial_Category___C___Non_HHI___Persons,Industrial_Category___C___Non_HHI___Males,Industrial_Category___C___Non_HHI___Females,Industrial_Category___D___E___Persons,Industrial_Category___D___E___Males,Industrial_Category___D___E___Females,Industrial_Category___F___Persons,Industrial_Category___F___Males,Industrial_Category___F___Females,Industrial_Category___G___HHI___Persons,Industrial_Category___G___HHI___Males,Industrial_Category___G___HHI___Females,Industrial_Category___G___Non_HHI___Persons,Industrial_Category___G___Non_HHI___Males,Industrial_Category___G___Non_HHI___Females,Industrial_Category___H___Persons,Industrial_Category___H___Males,Industrial_Category___H___Females,Industrial_Category___I___Persons,Industrial_Category___I___Males,Industrial_Category___I___Females,Industrial_Category___J___HHI___Persons,Industrial_Category___J___HHI___Males,Industrial_Category___J___HHI___Females,Industrial_Category___J___Non_HHI___Persons,Industrial_Category___J___Non_HHI___Males,Industrial_Category___J___Non_HHI___Females,Industrial_Category___K_to_M___Persons,Industrial_Category___K_to_M___Males,Industrial_Category___K_to_M___Females,Industrial_Category___N_to_O___Persons,Industrial_Category___N_to_O___Males,Industrial_Category___N_to_O___Females,Industrial_Category___P_to_Q___Persons,Industrial_Category___P_to_Q___Males,Industrial_Category___P_to_Q___Females,Industrial_Category___R_to_U___HHI___Persons,Industrial_Category___R_to_U___HHI___Males,Industrial_Category___R_to_U___HHI___Females,Industrial_Category___R_to_U___Non_HHI___Persons,Industrial_Category___R_to_U___Non_HHI___Males,Industrial_Category___R_to_U___Non_HHI___Females
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
Note: Table based on estimates.,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
INDUSTRIAL CATEGORIES :,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"A- Agriculture, Forestry and Fishing; B - Mining and Quarrying; C - Manufacturing; D - Electricity, Gas, Steam, and Air Conditioning Supply; E- Water Supply; (Sewarage, Waste Management and remediation activities); F - Construction; G- Wholesale and Retail Trade (Repair of motor vehicles and motor cycles); H- Transportation and Storage; I- Accomodation and food service activities;

In [0]:
display(investigate_invalid_rows(df_ind_main))

INVALID ROW INVESTIGATION
Rows containing at least one NULL : 5
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---

| Table                       |                Duplicate Rows | Null Rows | Gender Errors | Status                   |
| --------------------------- | ----------------------------: | --------: | ------------: | ------------------------ |
| st_main_workers             |                             0 |         0 |             0 | PASS                     |
| sc_main_workers             |                             0 |         0 |             0 | PASS                     |
| industrial_main_workers     | 1 duplicate + 5 metadata rows |         5 |             0 | PASS – Remove 5 metadata rows during Silver transformation |
| industrial_marginal_workers |                             0 |         0 |             0 | PASS                     |


| Table                   |    Duplicate Rows | Metadata Rows | Status              |
| ----------------------- | ----------------: | ------------: | ------------------- |
| industrial_main_workers | 0 data duplicates |             5 | Clean during Silver |
